In [ ]:
import os
import json
import pandas as pd

def parse_output_file(file_path):
    print(file_path)
    result = {}
    with open(file_path, 'r') as file:
        for line in file.readlines():
            print("Line: ", line)
            line = line.strip()
            tokens = line.split(":")
            result[tokens[0].strip()] = tokens[1].strip()
    if 'Query throughput' in result:
        result['query_thput'] = float(result['Query throughput'].split()[0])
    if 'False positives' in result:
        result['fp'] = float(result['False positives'].split()[0])
    if 'False positive rate' in result:
        result['fpr'] = float(result['False positive rate'].split()[0])
    if 'Filter throughput' in result:
        result['filter_thput'] = float(result['Filter throughput'].split()[0])
    if 'Filtered DB throughput' in result:
        result['db_thput'] = float(result['Filtered DB throughput'].split()[0])
    if 'ReverseMap DB throughput' in result:
        result['rm_thput'] = float(result['ReverseMap DB throughput'].split()[0])
    return result





In [ ]:
result_dir = "./breakEven"
experiment_runs = []
for dir in os.listdir(result_dir):
    if dir == "_sources":
        continue
    experiment_run = {}
    experiment_run['id'] = str(dir)
    with open(os.path.join(result_dir, dir, 'config.json'), 'r', encoding='utf-8') as file:
        experiment_run['config'] = json.load(file)
    with open(os.path.join(result_dir, dir, 'run.json'), 'r', encoding='utf-8') as file:
        experiment_run['context'] = json.load(file)
    experiment_run['result'] = parse_output_file(os.path.join(result_dir, dir, 'output.txt'))
    experiment_runs.append(experiment_run)

df = pd.json_normalize(experiment_runs)
df['result.breakeven'] = df['result.db_thput']/df['result.rm_thput']
print(df.columns)
display(df)

./breakEven/1/output.txt
Line:  Adaptive (filtered, batched) throughput:

Line:  Number of inserts:     3774873

Line:  Number of updates:     3774873

Line:  Time for inserts:      8.367292

Line:  Insert throughput:     451146.320697 ops/sec

Line:  CPU time for inserts:  8.164280

Line:  Time for queries:     0.109509 s

Line:  Filter throughput:     27543656.695863 ops/sec

Line:  Filtered DB throughput:     63790.270734 ops/sec

Line:  ReverseMap DB throughput:     15083.440308 ops/sec

Line:  Query throughput:     9131669.543143 ops/sec

Line:  False positives:      893

Line:  Num DB Queries:      893

Line:  False positive rate:  0.089300%

Index(['id', 'config.filter', 'config.num_queries', 'config.quotient_bits',
       'config.remainder_bits', 'config.seed', 'context.artifacts',
       'context.command', 'context.experiment.base_dir',
       'context.experiment.dependencies', 'context.experiment.mainfile',
       'context.experiment.name', 'context.experiment.repositories',


,id,config.filter,config.num_queries,config.quotient_bits,config.remainder_bits,config.seed,context.artifacts,context.command,context.experiment.base_dir,context.experiment.dependencies,...,result.Filtered DB throughput,result.ReverseMap DB throughput,result.Query throughput,result.False positives,result.Num DB Queries,result.False positive rate,result.query_thput,result.fp,result.db_thput,result.rm_thput
0,1,adaptive,1000000,22,10,740341074,[output.txt],run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,"[numpy==1.24.4, sacred==0.8.7]",...,63790.270734 ops/sec,15083.440308 ops/sec,9131669.543143 ops/sec,893,893,0.089300%,9.131670e+06,893.0,63790.270734,15083.440308


### Breakeven cost


In [ ]:
filtered_df = df[df['config.filter']=='adaptive']
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter'])
        .agg({
            'result.filter_thput': ['mean']
            'result.fp'
            'result.db_thput': ['mean'],
            'result.rm_thput': ['mean']
            'result.breakeven': ['mean']
            }))

,,,,result.db_thput,result.rm_thput
,,,,mean,mean
config.quotient_bits,config.remainder_bits,config.num_queries,config.filter,,
22,10,1000000,adaptive,63790.270734,15083.440308


### Uniform query

In [49]:
filtered_df = df[df['config.distribution']=='u'].dropna(subset=['result.fp'])
print(df[df['config.distribution']=='u'][['id','result.fp']].dropna())
#display(filtered_df)
#df['result.False positives']
#display(filtered_df[['config.filter','config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'result.query_thput', 'result.False positives']])
#display(filtered_df[['config.filter','config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'result.query_thput', 'result.False positives']])
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.query_thput': ['mean', 'min', 'max']}))
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.fp': ['mean', 'min', 'max']}))

    id  result.fp
0   54    87840.0
3   14    35035.0
11  67    87407.0
13  87    88039.0
17  84    87918.0
19  57    87860.0
20   8    34682.0
25  75    88213.0
37  64    87454.0
42  37    88383.0
43  34    88110.0
58  11    34977.0
61  24    87813.0
63  78    88029.0
66   4    35237.0
67  44    87714.0
70  27    88160.0
75  71    35252.0
80  81    88008.0
87  47    87491.0


result.query_thput  \
                                                                                          mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                      
24                   8                     10000000           DAdaptive           1.024083e+07   
                                                              adaptive            1.486665e+06   
                                                              nonAdaptive         1.168996e+07   
26                   10                    100000000          DAdaptive           4.865539e+06   
                                                              adaptive            1.811743e+06   
                                                              nonAdaptive         5.116886e+06   

                                                                                           \
                                                                                      min   
config.quotient_bits config.remainder_bits config.num_queries config.filter                 
24                   8                     10000000           DAdaptive      9.521080e+06   
                                                              adaptive       1.486665e+06   
                                                              nonAdaptive    1.168996e+07   
26                   10                    100000000          DAdaptive      3.763357e+06   
                                                              adaptive       1.718785e+06   
                                                              nonAdaptive    4.595593e+06   

                                                                                           
                                                                                      max  
config.quotient_bits config.remainder_bits config.num_queries config.filter                
24                   8                     10000000           DAdaptive      1.108510e+07  
                                                              adaptive       1.486665e+06  
                                                              nonAdaptive    1.168996e+07  
26                   10                    100000000          DAdaptive      5.555160e+06  
                                                              adaptive       1.877817e+06  
                                                              nonAdaptive    5.659180e+06

result.fp  \
                                                                                 mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter             
24                   8                     10000000           DAdaptive       35057.0   
                                                              adaptive        35035.0   
                                                              nonAdaptive     34977.0   
26                   10                    100000000          DAdaptive       88041.4   
                                                              adaptive        87860.2   
                                                              nonAdaptive     87786.2   

                                                                                      \
                                                                                 min   
config.quotient_bits config.remainder_bits config.num_queries config.filter            
24                   8                     10000000           DAdaptive      34682.0   
                                                              adaptive       35035.0   
                                                              nonAdaptive    34977.0   
26                   10                    100000000          DAdaptive      87918.0   
                                                              adaptive       87407.0   
                                                              nonAdaptive    87454.0   

                                                                                      
                                                                                 max  
config.quotient_bits config.remainder_bits config.num_queries config.filter           
24                   8                     10000000           DAdaptive      35252.0  
                                                              adaptive       35035.0  
                                                              nonAdaptive    34977.0  
26                   10                    100000000          DAdaptive      88213.0  
                                                              adaptive       88383.0  
                                                              nonAdaptive    88110.0

### Zipfian Distribution


In [ ]:
filtered_df = df[df['config.distribution']=='z']
display(filtered_df[['id', 'config.filter', 'result.False positives', 'result.query_thput']])
#df['result.False positives']
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.query_thput': ['mean', 'min', 'max'], 'result.fp': ['mean']}));


,id,config.filter,result.False positives,result.query_thput
1,10,DAdaptive,272,1.167692e+08
2,77,DAdaptive,279,8.285450e+07
4,49,adaptive,196,1.347456e+08
5,83,DAdaptive,284,8.315918e+07
8,80,DAdaptive,247,7.524998e+07
12,59,adaptive,191,1.312653e+08
23,56,nonAdaptive,4255,1.033855e+08
26,26,nonAdaptive,6449,1.107685e+08
29,43,DAdaptive,NaN,NaN
35,63,DAdaptive,NaN,NaN


result.query_thput  \
                                                                                          mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                      
24                   8                     10000000           DAdaptive           7.648069e+07   
                                                              adaptive            8.399268e+07   
                                                              nonAdaptive         9.131002e+07   
26                   10                    100000000          DAdaptive           1.144514e+08   
                                                              adaptive            1.269355e+08   
                                                              nonAdaptive         9.274354e+07   

                                                                                           \
                                                                                      min   
config.quotient_bits config.remainder_bits config.num_queries config.filter                 
24                   8                     10000000           DAdaptive      3.619215e+07   
                                                              adaptive       8.399268e+07   
                                                              nonAdaptive    9.131002e+07   
26                   10                    100000000          DAdaptive      7.524998e+07   
                                                              adaptive       9.670600e+07   
                                                              nonAdaptive    5.770770e+07   

                                                                                           
                                                                                      max  
config.quotient_bits config.remainder_bits config.num_queries config.filter                
24                   8                     10000000           DAdaptive      1.167692e+08  
                                                              adaptive       8.399268e+07  
                                                              nonAdaptive    9.131002e+07  
26                   10                    100000000          DAdaptive      1.698719e+08  
                                                              adaptive       1.363111e+08  
                                                              nonAdaptive    1.107685e+08

result.fp  \
                                                                                 mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter             
24                   8                     10000000           DAdaptive         264.0   
                                                              adaptive          198.0   
                                                              nonAdaptive      3391.0   
26                   10                    100000000          DAdaptive         269.8   
                                                              adaptive          189.0   
                                                              nonAdaptive      8589.4   

                                                                                     \
                                                                                min   
config.quotient_bits config.remainder_bits config.num_queries config.filter           
24                   8                     10000000           DAdaptive       256.0   
                                                              adaptive        198.0   
                                                              nonAdaptive    3391.0   
26                   10                    100000000          DAdaptive       247.0   
                                                              adaptive        176.0   
                                                              nonAdaptive    4255.0   

                                                                                      
                                                                                 max  
config.quotient_bits config.remainder_bits config.num_queries config.filter           
24                   8                     10000000           DAdaptive        272.0  
                                                              adaptive         198.0  
                                                              nonAdaptive     3391.0  
26                   10                    100000000          DAdaptive        284.0  
                                                              adaptive         203.0  
                                                              nonAdaptive    14952.0

### Adversarial test

In [53]:

filtered_df = df[df['config.distribution']=='a']
#display(filtered_df)
#df['result.False positives']
filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.query_thput': ['mean', 'min', 'max']})
filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.query_thput': ['mean', 'min', 'max'], 'result.fp': ['mean']})

result.query_thput  \
                                                                                          mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                      
24                   8                     10000000           DAdaptive           1.623100e+06   
                                                              adaptive            1.613166e+06   
                                                              nonAdaptive         3.437145e+06   
26                   10                    100000000          DAdaptive           2.050832e+06   
                                                              adaptive            2.103931e+06   
                                                              nonAdaptive         9.729771e+05   

                                                                                           \
                                                                                      min   
config.quotient_bits config.remainder_bits config.num_queries config.filter                 
24                   8                     10000000           DAdaptive      1.618845e+06   
                                                              adaptive       1.613166e+06   
                                                              nonAdaptive    3.437145e+06   
26                   10                    100000000          DAdaptive      1.938000e+06   
                                                              adaptive       2.069227e+06   
                                                              nonAdaptive    8.539027e+05   

                                                                                           \
                                                                                      max   
config.quotient_bits config.remainder_bits config.num_queries config.filter                 
24                   8                     10000000           DAdaptive      1.627355e+06   
                                                              adaptive       1.613166e+06   
                                                              nonAdaptive    3.437145e+06   
26                   10                    100000000          DAdaptive      2.131646e+06   
                                                              adaptive       2.141130e+06   
                                                              nonAdaptive    1.159309e+06   

                                                                             result.fp  
                                                                                  mean  
config.quotient_bits config.remainder_bits config.num_queries config.filter             
24                   8                     10000000           DAdaptive        34919.5  
                                                              adaptive         35080.0  
                                                              nonAdaptive     533131.0  
26                   10                    100000000          DAdaptive        87677.0  
                                                              adaptive         87860.0  
                                                              nonAdaptive    5083356.6